In [ ]:
! pip install chembl_webresource_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

In [ ]:
target = new_client.target
target_query = target.search('HER2')
targets = pd.DataFrame.from_dict(target_query)
targets

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,FASN/HER2,17.0,False,CHEMBL4106134,"[{'accession': 'P04626', 'component_descriptio...",PROTEIN COMPLEX,9606
1,"[{'xref_id': 'P04626', 'xref_name': None, 'xre...",Homo sapiens,Receptor protein-tyrosine kinase erbB-2,15.0,False,CHEMBL1824,"[{'accession': 'P04626', 'component_descriptio...",SINGLE PROTEIN,9606
2,[],Homo sapiens,Epidermal growth factor receptor and ErbB2 (HE...,13.0,False,CHEMBL2111431,"[{'accession': 'P04626', 'component_descriptio...",PROTEIN FAMILY,9606
3,[],Homo sapiens,ErbB-2/ErbB-3 heterodimer,12.0,False,CHEMBL4630723,"[{'accession': 'P04626', 'component_descriptio...",PROTEIN COMPLEX,9606
4,[],Homo sapiens,Epidermal growth factor receptor,9.0,False,CHEMBL2363049,"[{'accession': 'P04626', 'component_descriptio...",PROTEIN FAMILY,9606
5,"[{'xref_id': 'P06494', 'xref_name': None, 'xre...",Rattus norvegicus,Receptor protein-tyrosine kinase erbB-2,6.0,False,CHEMBL3848,"[{'accession': 'P06494', 'component_descriptio...",SINGLE PROTEIN,10116


In [ ]:
selected_target = targets.target_chembl_id[1]
selected_target

'CHEMBL1824'

In [ ]:
activity = new_client.activity
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")

In [ ]:
df = pd.DataFrame.from_dict(res)

In [ ]:
df.head(3)

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,32264,[],CHEMBL845865,Inhibition of autophosphorylation of human Her...,F,None,None,BAO_0000190,...,Homo sapiens,Receptor protein-tyrosine kinase erbB-2,9606,None,None,IC50,uM,UO_0000065,None,0.3
1,None,None,32266,[],CHEMBL615491,Inhibition of ligand induced proliferation in ...,F,None,None,BAO_0000190,...,Homo sapiens,Receptor protein-tyrosine kinase erbB-2,9606,None,None,IC50,uM,UO_0000065,None,2.5
2,None,None,32271,[],CHEMBL683802,Inhibition of autophosphorylation of human Her...,F,None,None,BAO_0000190,...,Homo sapiens,Receptor protein-tyrosine kinase erbB-2,9606,None,None,IC50,uM,UO_0000065,None,0.4


In [ ]:
df.target_pref_name.unique()

array(['Receptor protein-tyrosine kinase erbB-2'], dtype=object)

In [ ]:
df.units.unique()

array(['uM', 'nM', None, 'mM', 'M', 'ug ml-1', "10'-6g/ml", "10'-5g/ml",
       'nmol/L', "10'-6M", "10'-7M", '10^-8M', 'nmol'], dtype=object)

In [ ]:
df.units.value_counts()

,count
units,
uM,1658
nM,1176
ug ml-1,34
10'-5g/ml,6
10'-6g/ml,4
nmol,3
nmol/L,2
mM,1
M,1


In [ ]:
df_sel=df.loc[df['units']=='uM']

In [ ]:
df_sel.units.value_counts()

,count
units,
uM,1658


In [ ]:
len(df_sel)

1658

Data Cleaning

Missing values

In [ ]:
df2 = df_sel[df_sel.standard_value.notna()]
df2 = df2[df_sel.canonical_smiles.notna()]

<ipython-input-14-4cd1e99d5d9d>:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df2 = df2[df_sel.canonical_smiles.notna()]


In [ ]:
df2.head(3)

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,32264,[],CHEMBL845865,Inhibition of autophosphorylation of human Her...,F,None,None,BAO_0000190,...,Homo sapiens,Receptor protein-tyrosine kinase erbB-2,9606,None,None,IC50,uM,UO_0000065,None,0.3
1,None,None,32266,[],CHEMBL615491,Inhibition of ligand induced proliferation in ...,F,None,None,BAO_0000190,...,Homo sapiens,Receptor protein-tyrosine kinase erbB-2,9606,None,None,IC50,uM,UO_0000065,None,2.5
2,None,None,32271,[],CHEMBL683802,Inhibition of autophosphorylation of human Her...,F,None,None,BAO_0000190,...,Homo sapiens,Receptor protein-tyrosine kinase erbB-2,9606,None,None,IC50,uM,UO_0000065,None,0.4


In [ ]:
len(df2.canonical_smiles.unique())

1482

In [ ]:
len(df2)

1633

In [ ]:
df2_nr = df2.drop_duplicates(['canonical_smiles'])

In [ ]:

len(df2_nr)

1482

In [ ]:
len(df2_nr)

1482

In [ ]:
selection = ['molecule_chembl_id','canonical_smiles','standard_value','units']
df3 = df2_nr[selection]

In [ ]:
df3

,molecule_chembl_id,canonical_smiles,standard_value,units
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,uM
2,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,400.0,uM
4,CHEMBL67057,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,100.0,uM
6,CHEMBL65848,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,5000.0,uM
8,CHEMBL69629,Cc1cc(C(=O)NCCN2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncn...,100.0,uM
...,...,...,...,...
4087,CHEMBL4539558,N=C1NC(=O)/C(=C2/CCNC(=O)c3[nH]cc(Br)c32)N1,18460.0,uM
4088,CHEMBL1095259,N=C1N=C(NCCS(=O)(=O)O)/C(=C/C(O)CNC(=O)c2cc(Br...,37930.0,uM
4089,CHEMBL4575427,N=C1N=C(NCCS(=O)(=O)O)/C(=C/C(O)CNC(=O)c2cc(Br...,37930.0,uM
4090,CHEMBL4104297,Nc1nccc(Oc2ccc3c(C(=O)Nc4cccc(C(F)(F)F)c4)cccc...,500.0,uM


In [ ]:
df3.units.value_counts()

,count
units,
uM,1482


In [ ]:
df3.to_csv('HER2_02_bioactivity_data_preprocessed.csv', index=False)

Labelling Active and inactive

In [ ]:
df4 = pd.read_csv('HER2_02_bioactivity_data_preprocessed.csv')

In [ ]:
bioactivity_threshold = []
for i in df4.standard_value:
  if float(i) <= 1000:
    bioactivity_threshold.append("active")
  #elif float(i) >= 10000:
      #bioactivity_threshold.append("inactive")
  else:
    bioactivity_threshold.append("inactive")

In [ ]:
bioactivity_class = pd.Series(bioactivity_threshold, name='activity')
df5 = pd.concat([df4, bioactivity_class], axis=1)
df5

,molecule_chembl_id,canonical_smiles,standard_value,units,activity
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,uM,active
1,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,400.0,uM,active
2,CHEMBL67057,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,100.0,uM,active
3,CHEMBL65848,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,5000.0,uM,inactive
4,CHEMBL69629,Cc1cc(C(=O)NCCN2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncn...,100.0,uM,active
...,...,...,...,...,...
1477,CHEMBL4539558,N=C1NC(=O)/C(=C2/CCNC(=O)c3[nH]cc(Br)c32)N1,18460.0,uM,inactive
1478,CHEMBL1095259,N=C1N=C(NCCS(=O)(=O)O)/C(=C/C(O)CNC(=O)c2cc(Br...,37930.0,uM,inactive
1479,CHEMBL4575427,N=C1N=C(NCCS(=O)(=O)O)/C(=C/C(O)CNC(=O)c2cc(Br...,37930.0,uM,inactive
1480,CHEMBL4104297,Nc1nccc(Oc2ccc3c(C(=O)Nc4cccc(C(F)(F)F)c4)cccc...,500.0,uM,active


In [ ]:
df5.to_csv('Her2_03_bioactivity_data_curated.csv', index=False)

In [ ]:
df5.isnull().sum()

,0
molecule_chembl_id,0
canonical_smiles,0
standard_value,0
units,0
activity,0


In [ ]:
df5['activity'].value_counts()

,count
activity,
inactive,759
active,723


Labelling the DATA Active=0, inactive =1

In [ ]:
from sklearn import preprocessing
from sklearn.preprocessing import LabelEncoder
label_encoder = preprocessing.LabelEncoder()


df5['activity']= label_encoder.fit_transform(df5['activity'])

In [ ]:
df5['activity'].value_counts()

,count
activity,
1,759
0,723


In [ ]:
df6 =  df5['canonical_smiles']
df6.to_csv('molecule.smi', sep='\t', index=False, header=False)
df6

,canonical_smiles
0,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...
1,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...
2,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...
3,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...
4,Cc1cc(C(=O)NCCN2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncn...
...,...
1477,N=C1NC(=O)/C(=C2/CCNC(=O)c3[nH]cc(Br)c32)N1
1478,N=C1N=C(NCCS(=O)(=O)O)/C(=C/C(O)CNC(=O)c2cc(Br...
1479,N=C1N=C(NCCS(=O)(=O)O)/C(=C/C(O)CNC(=O)c2cc(Br...
1480,Nc1nccc(Oc2ccc3c(C(=O)Nc4cccc(C(F)(F)F)c4)cccc...


In [ ]:
len(df6)

1482

FingerPrints

In [ ]:
! pip install padelpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 26.6 MB/s eta 0:00:00


In [ ]:
! wget https://github.com/dataprofessor/padel/raw/main/fingerprints_xml.zip
! unzip fingerprints_xml.zip

--2024-12-04 11:47:42--  https://github.com/dataprofessor/padel/raw/main/fingerprints_xml.zip
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/dataprofessor/padel/main/fingerprints_xml.zip [following]
--2024-12-04 11:47:42--  https://raw.githubusercontent.com/dataprofessor/padel/main/fingerprints_xml.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10871 (11K) [application/zip]
Saving to: ‘fingerprints_xml.zip’

fingerprints_xml.zi 100%[===================>]  10.62K  --.-KB/s    in 0s      

2024-12-04 11:47:43 (38.7 MB/s) - ‘fingerprints_xml.zip’ saved [10871/10871]

Archive:  fingerprints_xm

In [ ]:
import glob
xml_files = glob.glob("*.xml")
xml_files.sort()
xml_files

['AtomPairs2DFingerprintCount.xml',
 'AtomPairs2DFingerprinter.xml',
 'EStateFingerprinter.xml',
 'ExtendedFingerprinter.xml',
 'Fingerprinter.xml',
 'GraphOnlyFingerprinter.xml',
 'KlekotaRothFingerprintCount.xml',
 'KlekotaRothFingerprinter.xml',
 'MACCSFingerprinter.xml',
 'PubchemFingerprinter.xml',
 'SubstructureFingerprintCount.xml',
 'SubstructureFingerprinter.xml']

In [ ]:
FP_list = ['AtomPairs2DCount',
 'AtomPairs2D',
 'EState',
 'CDKextended',
 'CDK',
 'CDKgraphonly',
 'KlekotaRothCount',
 'KlekotaRoth',
 'MACCS',
 'PubChem',
 'SubstructureCount',
 'Substructure']

Dictionary Creation

In [ ]:
fp = dict(zip(FP_list, xml_files))
fp

{'AtomPairs2DCount': 'AtomPairs2DFingerprintCount.xml',
 'AtomPairs2D': 'AtomPairs2DFingerprinter.xml',
 'EState': 'EStateFingerprinter.xml',
 'CDKextended': 'ExtendedFingerprinter.xml',
 'CDK': 'Fingerprinter.xml',
 'CDKgraphonly': 'GraphOnlyFingerprinter.xml',
 'KlekotaRothCount': 'KlekotaRothFingerprintCount.xml',
 'KlekotaRoth': 'KlekotaRothFingerprinter.xml',
 'MACCS': 'MACCSFingerprinter.xml',
 'PubChem': 'PubchemFingerprinter.xml',
 'SubstructureCount': 'SubstructureFingerprintCount.xml',
 'Substructure': 'SubstructureFingerprinter.xml'}

In [ ]:
fp['KlekotaRoth'] # key & word

'KlekotaRothFingerprinter.xml'

In [ ]:
df5.head(3)

,molecule_chembl_id,canonical_smiles,standard_value,units,activity
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,uM,0
1,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,400.0,uM,0
2,CHEMBL67057,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,100.0,uM,0


In [ ]:
fp

{'AtomPairs2DCount': 'AtomPairs2DFingerprintCount.xml',
 'AtomPairs2D': 'AtomPairs2DFingerprinter.xml',
 'EState': 'EStateFingerprinter.xml',
 'CDKextended': 'ExtendedFingerprinter.xml',
 'CDK': 'Fingerprinter.xml',
 'CDKgraphonly': 'GraphOnlyFingerprinter.xml',
 'KlekotaRothCount': 'KlekotaRothFingerprintCount.xml',
 'KlekotaRoth': 'KlekotaRothFingerprinter.xml',
 'MACCS': 'MACCSFingerprinter.xml',
 'PubChem': 'PubchemFingerprinter.xml',
 'SubstructureCount': 'SubstructureFingerprintCount.xml',
 'Substructure': 'SubstructureFingerprinter.xml'}

MACCS

In [ ]:
from padelpy import padeldescriptor

fingerprint = 'MACCS'

fingerprint_output_file = ''.join([fingerprint,'.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='molecule.smi',
                d_file=fingerprint_output_file, #'Substructure.csv'

                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)


In [ ]:
df6mc = pd.read_csv(fingerprint_output_file)
df6mc

,Name,MACCSFP1,MACCSFP2,MACCSFP3,MACCSFP4,MACCSFP5,MACCSFP6,MACCSFP7,MACCSFP8,MACCSFP9,...,MACCSFP157,MACCSFP158,MACCSFP159,MACCSFP160,MACCSFP161,MACCSFP162,MACCSFP163,MACCSFP164,MACCSFP165,MACCSFP166
0,AUTOGEN_molecule_1,0,0,0,0,0,0,0,0,0,...,0,1,0,1,1,1,1,1,1,0
1,AUTOGEN_molecule_2,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
2,AUTOGEN_molecule_3,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
3,AUTOGEN_molecule_4,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
4,AUTOGEN_molecule_5,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,AUTOGEN_molecule_1478,0,0,0,0,0,0,0,0,0,...,0,1,1,0,1,1,0,1,1,0
1478,AUTOGEN_molecule_1479,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,0,1,1,0
1479,AUTOGEN_molecule_1480,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,0,1,1,0
1480,AUTOGEN_molecule_1481,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1,1,1,1,1,0


In [ ]:
dfmaccs = pd.concat([df5,df6mc], axis=1)
dfmaccs.head(3)

,molecule_chembl_id,canonical_smiles,standard_value,units,activity,Name,MACCSFP1,MACCSFP2,MACCSFP3,MACCSFP4,...,MACCSFP157,MACCSFP158,MACCSFP159,MACCSFP160,MACCSFP161,MACCSFP162,MACCSFP163,MACCSFP164,MACCSFP165,MACCSFP166
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,uM,0,AUTOGEN_molecule_1,0,0,0,0,...,0,1,0,1,1,1,1,1,1,0
1,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,400.0,uM,0,AUTOGEN_molecule_2,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
2,CHEMBL67057,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,100.0,uM,0,AUTOGEN_molecule_3,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0


In [ ]:
len(dfmaccs)

1482

In [ ]:
dfmaccs.to_csv('dfmaccs_her2.csv')

PubChem

In [ ]:
from padelpy import padeldescriptor

fingerprint = 'PubChem'

fingerprint_output_file = ''.join([fingerprint,'.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='molecule.smi',
                d_file=fingerprint_output_file, #'Substructure.csv'

                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)


In [ ]:
df6pub = pd.read_csv(fingerprint_output_file)
df6pub


,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,AUTOGEN_molecule_1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AUTOGEN_molecule_2,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,AUTOGEN_molecule_3,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,AUTOGEN_molecule_4,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,AUTOGEN_molecule_5,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,AUTOGEN_molecule_1478,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1478,AUTOGEN_molecule_1479,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1479,AUTOGEN_molecule_1480,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1480,AUTOGEN_molecule_1481,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_pubchem = pd.concat([df5,df6pub], axis=1)
df_pubchem.head(3)


,molecule_chembl_id,canonical_smiles,standard_value,units,activity,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,uM,0,AUTOGEN_molecule_1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,400.0,uM,0,AUTOGEN_molecule_2,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL67057,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,100.0,uM,0,AUTOGEN_molecule_3,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
len(df_pubchem)

1482

In [ ]:
df_pubchem.to_csv('dfpubchem_her2.csv')

Estate

In [ ]:
from padelpy import padeldescriptor

fingerprint = 'EState'

fingerprint_output_file = ''.join([fingerprint,'.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='molecule.smi',
                d_file=fingerprint_output_file, #'Substructure.csv'

                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)


In [ ]:
df6est = pd.read_csv(fingerprint_output_file)
df6est


,Name,EStateFP1,EStateFP2,EStateFP3,EStateFP4,EStateFP5,EStateFP6,EStateFP7,EStateFP8,EStateFP9,...,EStateFP70,EStateFP71,EStateFP72,EStateFP73,EStateFP74,EStateFP75,EStateFP76,EStateFP77,EStateFP78,EStateFP79
0,AUTOGEN_molecule_1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AUTOGEN_molecule_2,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,AUTOGEN_molecule_3,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
3,AUTOGEN_molecule_4,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,AUTOGEN_molecule_5,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,AUTOGEN_molecule_1478,0,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
1478,AUTOGEN_molecule_1479,0,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
1479,AUTOGEN_molecule_1480,0,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
1480,AUTOGEN_molecule_1481,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_estate = pd.concat([df5,df6est], axis=1)
df_estate.head(3)


,molecule_chembl_id,canonical_smiles,standard_value,units,activity,Name,EStateFP1,EStateFP2,EStateFP3,EStateFP4,...,EStateFP70,EStateFP71,EStateFP72,EStateFP73,EStateFP74,EStateFP75,EStateFP76,EStateFP77,EStateFP78,EStateFP79
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,uM,0,AUTOGEN_molecule_1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,400.0,uM,0,AUTOGEN_molecule_2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL67057,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,100.0,uM,0,AUTOGEN_molecule_3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
len(df_estate)

1482

In [ ]:
df_estate.to_csv('dfestate_her2.csv')

AtomsPairs2D

In [ ]:
from padelpy import padeldescriptor

fingerprint = 'AtomPairs2D'

fingerprint_output_file = ''.join([fingerprint,'.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='molecule.smi',
                d_file=fingerprint_output_file, #'Substructure.csv'

                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)



In [ ]:
df6atom = pd.read_csv(fingerprint_output_file)
df6atom


,Name,AD2D1,AD2D2,AD2D3,AD2D4,AD2D5,AD2D6,AD2D7,AD2D8,AD2D9,...,AD2D771,AD2D772,AD2D773,AD2D774,AD2D775,AD2D776,AD2D777,AD2D778,AD2D779,AD2D780
0,AUTOGEN_molecule_1,1,1,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,AUTOGEN_molecule_2,1,1,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,AUTOGEN_molecule_3,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,AUTOGEN_molecule_4,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,AUTOGEN_molecule_5,1,1,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,AUTOGEN_molecule_1478,1,1,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1478,AUTOGEN_molecule_1479,1,1,1,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1479,AUTOGEN_molecule_1480,1,1,1,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1480,AUTOGEN_molecule_1481,1,1,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_atompair = pd.concat([df5,df6atom], axis=1)
df_atompair.head(3)


,molecule_chembl_id,canonical_smiles,standard_value,units,activity,Name,AD2D1,AD2D2,AD2D3,AD2D4,...,AD2D771,AD2D772,AD2D773,AD2D774,AD2D775,AD2D776,AD2D777,AD2D778,AD2D779,AD2D780
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,uM,0,AUTOGEN_molecule_1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,400.0,uM,0,AUTOGEN_molecule_2,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL67057,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,100.0,uM,0,AUTOGEN_molecule_3,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
len(df_atompair)

1482

In [ ]:
df_atompair.to_csv('dfatompair_her2.csv')

Substructure

In [ ]:
from padelpy import padeldescriptor

fingerprint = 'Substructure'

fingerprint_output_file = ''.join([fingerprint,'.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='molecule.smi',
                d_file=fingerprint_output_file, #'Substructure.csv'

                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)


In [ ]:
df6sub = pd.read_csv(fingerprint_output_file)
df6sub


,Name,SubFP1,SubFP2,SubFP3,SubFP4,SubFP5,SubFP6,SubFP7,SubFP8,SubFP9,...,SubFP298,SubFP299,SubFP300,SubFP301,SubFP302,SubFP303,SubFP304,SubFP305,SubFP306,SubFP307
0,AUTOGEN_molecule_1,1,0,0,0,1,0,0,0,0,...,0,0,1,1,1,1,0,0,0,1
1,AUTOGEN_molecule_2,1,0,0,0,1,0,0,0,0,...,0,0,1,1,1,1,0,0,0,1
2,AUTOGEN_molecule_3,1,0,0,0,1,0,0,0,0,...,0,0,1,1,1,1,0,0,0,1
3,AUTOGEN_molecule_4,1,1,0,0,1,0,0,0,0,...,0,0,1,1,1,1,0,0,0,1
4,AUTOGEN_molecule_5,1,0,0,0,1,0,0,0,0,...,0,0,1,1,1,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,AUTOGEN_molecule_1478,0,1,1,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
1478,AUTOGEN_molecule_1479,0,1,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
1479,AUTOGEN_molecule_1480,0,1,0,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
1480,AUTOGEN_molecule_1481,0,0,0,0,0,0,0,0,1,...,0,0,1,1,1,0,0,0,0,1


In [ ]:
df_subst = pd.concat([df5,df6sub], axis=1)
df_subst.head(3)


,molecule_chembl_id,canonical_smiles,standard_value,units,activity,Name,SubFP1,SubFP2,SubFP3,SubFP4,...,SubFP298,SubFP299,SubFP300,SubFP301,SubFP302,SubFP303,SubFP304,SubFP305,SubFP306,SubFP307
0,CHEMBL68920,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,300.0,uM,0,AUTOGEN_molecule_1,1,0,0,0,...,0,0,1,1,1,1,0,0,0,1
1,CHEMBL69960,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,400.0,uM,0,AUTOGEN_molecule_2,1,0,0,0,...,0,0,1,1,1,1,0,0,0,1
2,CHEMBL67057,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,100.0,uM,0,AUTOGEN_molecule_3,1,0,0,0,...,0,0,1,1,1,1,0,0,0,1


In [ ]:
len(df_subst)

1482

In [ ]:
df_subst.to_csv('dfsubst_her2.csv')